### 从零实现一个神经网络训练系统

In [ ]:
# 数据构造
import torch
from torch import nn

torch.manual_seed(42)

x= torch.randn(512, 2)

y = (
    2 * x[:, 0:1]
    - 3 * x[:, 1:2]
    + 0.5
)

In [ ]:

device = torch.device(
    "cuda"if torch.cuda.is_available() else "cpu" 
)
x = x.to(device=device)
y = y.to(device=device)

class mylinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.randn(2,16))
        self.b = nn.Parameter(torch.zeros(16))
    def forward(self,x):
        return x@self.w+self.b
    
class buffer(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer(
    "input_offset",
    torch.tensor([0.5, -0.25])
)
    def forward(self,x):
        return x-self.input_offset
        
class MyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.buf = buffer()
        self.lin1 = mylinear()
        self.relu = nn.ReLU()
        self.lin2 = nn.Linear(16, 1)
        
    def forward(self, x):
        x = self.buf(x)
        x = self.lin1(x)
        x = self.relu(x)
        x = self.lin2(x)
        return x

net = MyNet().to(device)

for name,param in net.named_parameters():
    print(name, param.shape)

print(sum(p.numel() for p in net.parameters()))

epochs = 10
lr = 0.01
optimizer = torch.optim.Adam(net.parameters(),lr=lr)

loss = nn.MSELoss()
for epoch in range(epochs):
    optimizer.zero_grad()
    y_hat = net(x)
    l = loss(y_hat,y)
    l.backward()
    optimizer.step()
    print(l.item())
    
checkpoint = {
    "model":net.state_dict(),
    "epoch":epoch,
    "optimizer":optimizer.state_dict()
}
torch.save(checkpoint,"checkpoint.pth")

net2 = MyNet().to(device)
optimizer2 =  torch.optim.Adam(net2.parameters(),lr=lr)
checkpoint2 = torch.load("checkpoint.pth",map_location=device,weights_only=True)
net2.load_state_dict(checkpoint2["model"])
optimizer2.load_state_dict(checkpoint2["optimizer"])
epoch = checkpoint2["epoch"]+1

net.eval()
net2.eval()

y_hat1 = net(x)
y_hat2 = net2(x)

for epoch in range(190):
    optimizer2.zero_grad()
    y_hat = net2(x)
    l = loss(y_hat,y)
    l.backward()
    optimizer2.step()
    print(l.item())
    